# 💳 Credit Risk Scoring (Kaggle Style)
Binary classification for creditworthiness using LightGBM, SHAP, and Gradio UI.

In [ ]:
!pip install lightgbm shap gradio --quiet

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import shap
import gradio as gr


In [ ]:
np.random.seed(111)
n = 1500
df = pd.DataFrame({
    "income": np.random.randint(20000, 150000, n),
    "credit_score": np.random.randint(300, 850, n),
    "loan_amount": np.random.randint(5000, 100000, n),
    "loan_term": np.random.choice([12, 24, 36, 48, 60], n),
    "debt_to_income": np.random.uniform(0.1, 0.6, n),
    "default": np.random.choice([0, 1], n, p=[0.85, 0.15])
})


In [ ]:
X = df.drop(columns="default")
y = df["default"]
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=111)
model = lgb.LGBMClassifier()
model.fit(X_train, y_train)
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))


In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values[1], X_test)


In [ ]:
def predict_credit_risk(income, credit_score, loan_amount, loan_term, debt_to_income):
    row = pd.DataFrame([[income, credit_score, loan_amount, loan_term, debt_to_income]], columns=X.columns)
    prob = model.predict_proba(row)[0][1]
    return f"Default Risk: {prob:.2%}"

gr.Interface(
    fn=predict_credit_risk,
    inputs=[
        gr.Number(label="Income"),
        gr.Slider(300, 850, label="Credit Score"),
        gr.Number(label="Loan Amount"),
        gr.Dropdown([12, 24, 36, 48, 60], label="Loan Term (months)"),
        gr.Slider(0.0, 1.0, step=0.01, label="Debt-to-Income Ratio")
    ],
    outputs="text",
    title="Credit Risk Predictor"
).launch()
